# ShopFlow — Phase 0 Setup
### Complete Snowflake environment setup — from a brand new account to ready-to-load

---

**Who this notebook is for**

You just created a Snowflake account. You have never used Snowflake before. This Snowflake Notebook walks you through every single object you need to create — in the right order — before any data can be loaded.

By the end of this notebook you will have built this:

```text
your Snowflake account
└── SHOPFLOW_DB                        (database)
    ├── SHOPFLOW_RAW                   (schema — landing zone)
    │   ├── OLIST_CSV_FORMAT           (file format)
    │   ├── OLIST_STAGE                (internal stage)
    │   ├── RAW_ORDERS
    │   ├── RAW_ORDER_ITEMS
    │   ├── RAW_CUSTOMERS
    │   ├── RAW_SELLERS
    │   ├── RAW_PRODUCTS
    │   ├── RAW_PAYMENTS
    │   ├── RAW_REVIEWS
    │   ├── RAW_GEOLOCATION
    │   └── RAW_CATEGORY_TRANSLATION
    ├── SHOPFLOW_STAGED                (schema — Phase 2)
    └── SHOPFLOW_ANALYTICS             (schema — Phase 3)
```

---

**How to run this Snowflake Notebook in Snowsight**

1. Log into your Snowflake account at `app.snowflake.com`
2. In the left sidebar click **Projects → Notebooks**
3. Click **+ Notebook** (top right) → **Import .ipynb**
4. Upload this file
5. When prompted, select any available warehouse (you will create the right one in Cell 2)
6. Run cells **one at a time, top to bottom** — do not use Run All for this setup phase

---

**Dependency order — why sequence matters**

Every object in Snowflake lives inside another object. You cannot create a child before its parent exists:

```text
account  →  warehouse  →  database  →  schema  →  file format
                                               →  stage
                                               →  tables
```

Skip a step or run out of order and Snowflake will return an error. Follow the cells in sequence.

---

**🚀 Key Snowflake Concepts Covered in this Notebook**

| Cell | Object created | Domain Concept |
|------|---------------|-------------|
| 2 | Virtual warehouse | Architecture — compute vs storage |
| 3 | Database | Architecture — object hierarchy |
| 4 | Role + grants | Security — RBAC |
| 5 | Schemas (3) | Architecture — namespaces |
| 6 | File format | Data loading — COPY INTO options |
| 7 | Internal stage | Data loading — stages |
| 8–16 | RAW tables | Data loading — raw layer design |

---
## Cell 1 — Explore your brand new account

When you first create a Snowflake account, Snowflake provisions a few default objects for you automatically. Before building anything, run these cells to see exactly what already exists.

*(Note: In Snowflake Notebooks, only the last query in a cell displays its output grid. To let you see all outputs, we've split these into individual cells!)*

> **💡 Pro Tip — The Snowflake Object Hierarchy**
> 
> Snowflake organises everything in a strict hierarchy. Nothing can exist outside its parent:
> 
> ```text
> Organisation
>   └── Account           ← you are here
>         ├── Warehouses  ← compute (CPU/memory)
>         ├── Databases   ← storage containers
>         │     └── Schemas
>         │           └── Tables / Views / Stages / File Formats
>         └── Roles / Users
> ```
> 
> Notice that **warehouses and databases are siblings** — they are both directly under the account and completely independent of each other. A warehouse provides compute. A database stores data. They do not contain each other. This is a foundational architectural concept in Snowflake and is absolutely critical for mastering the platform.

In [ ]:
-- Who am I and what role am I using right now?
SELECT
    CURRENT_USER()      AS my_username,
    CURRENT_ROLE()      AS my_active_role,
    CURRENT_ACCOUNT()   AS my_account_locator,
    CURRENT_REGION()    AS my_region;

In [ ]:
-- What warehouses already exist in my account?
-- Look at the 'name', 'state', 'size', and 'auto_suspend' columns
SHOW WAREHOUSES;

In [ ]:
-- What databases already exist?
-- SNOWFLAKE and SNOWFLAKE_SAMPLE_DATA are created by Snowflake — do not modify them
SHOW DATABASES;

In [ ]:
-- What roles exist?
-- You will see the 5 built-in system roles
SHOW ROLES;

---
## Cell 2 — Create a virtual warehouse

### What is a virtual warehouse?

A virtual warehouse is **compute** — it is the cluster of CPU and memory that executes your SQL queries. Without a running warehouse, no query can run — not even a simple `SELECT 1`.

This is the biggest conceptual shift coming from traditional databases like MySQL or Postgres, where compute and storage are bundled together in one server. In Snowflake they are completely separate:

```text
Traditional database (e.g. RDS):          Snowflake:
┌─────────────────────────┐               ┌──────────────┐   ┌───────────────┐
│  Server                 │               │  Warehouse   │   │   Database    │
│  ├── CPU / Memory       │               │  (compute)   │   │   (storage)   │
│  └── Storage (data)     │               └──────────────┘   └───────────────┘
└─────────────────────────┘               completely separate — scale independently
```

### Why X-Small for this project?

Warehouse sizes go from X-Small → Small → Medium → Large → X-Large → 2X-Large → ... → 6X-Large. Each size up **doubles the compute and doubles the cost**.

X-Small is 1 credit per hour. For loading CSVs and running analytical queries on ~1 million rows, X-Small is more than enough. There is no reason to use a larger size — you would just burn through your free trial credits faster.

### AUTO_SUSPEND and AUTO_RESUME

A warehouse costs credits **only when it is running**. When idle it should suspend automatically.

- `AUTO_SUSPEND = 60` — the warehouse suspends itself after 60 seconds of inactivity. Zero credits charged while suspended.
- `AUTO_RESUME = TRUE` — the warehouse starts itself automatically the moment you run a query. You never need to manually start it.

This combination means you only pay for the seconds you are actually running queries. For a learning project this is critical — without AUTO_SUSPEND a warehouse left running overnight would drain your trial credits.

> **💡 Concept Note: Suspended Warehouses**
> 
> A common point of confusion is: *what happens when a warehouse is suspended?* 
> 
> **Answer: no queries can run, but data is not lost.** Storage (your databases and tables) is completely independent of compute (your warehouse). Suspending a warehouse does not affect your data in any way.

In [ ]:
-- Switch to SYSADMIN role — this role has privileges to create warehouses and databases
USE ROLE SYSADMIN;

-- Create a dedicated warehouse for the ShopFlow project
CREATE WAREHOUSE IF NOT EXISTS SHOPFLOW_WH
    WAREHOUSE_SIZE = 'X-SMALL'    -- smallest and cheapest — sufficient for this project
    AUTO_SUSPEND   = 60           -- suspend after 60 seconds idle — saves credits
    AUTO_RESUME    = TRUE         -- start automatically when a query is run
    COMMENT        = 'ShopFlow project warehouse — X-Small, auto-suspend 60s';

-- Activate the warehouse for this session
USE WAREHOUSE SHOPFLOW_WH;

-- Confirm warehouse is running
SHOW WAREHOUSES LIKE 'SHOPFLOW%';

---
## Cell 3 — Create a database

### What is a database in Snowflake?

A database in Snowflake is a **storage container** — it holds schemas, which hold tables, views, stages, and other objects. It does not contain compute (that is the warehouse's job).

Think of it like this:

```text
Snowflake database  ≈  AWS S3 bucket (a named container for your data)
Snowflake schema    ≈  S3 folder prefix (a namespace inside the container)
Snowflake table     ≈  S3 object / Redshift table (the actual data)
```

### Why create a new database instead of using an existing one?

Your Snowflake account already has `SNOWFLAKE` and `SNOWFLAKE_SAMPLE_DATA`. Do not use these — they are managed by Snowflake and exist for system purposes and sample data respectively.

Creating `SHOPFLOW_DB` gives the project a clean, isolated home. Everything related to ShopFlow — raw tables, staged tables, analytics tables, stages, file formats — lives under this one database. If you ever want to delete the entire project, you drop one database and everything is gone.

### DATA_RETENTION_TIME_IN_DAYS = 1

This enables **Time Travel** for 1 day. Time Travel lets you query data as it existed at any point in the past — for example, `SELECT * FROM my_table AT (TIMESTAMP => '2024-01-15 10:00:00')`. It also lets you recover accidentally dropped tables.

The trial account default is 1 day which is fine for learning. Production databases typically use 7–90 days.

> **⏱️ Pro Tip: Time Travel Limits**
> 
> Make sure to remember these limits: **Standard edition** supports up to 1 day. **Enterprise edition** supports up to 90 days. After Time Travel expires, data moves into **Fail-safe** — a 7-day emergency recovery period managed by Snowflake Support (you cannot query data in Fail-safe directly).

In [ ]:
-- Create the ShopFlow project database
CREATE DATABASE IF NOT EXISTS SHOPFLOW_DB
    DATA_RETENTION_TIME_IN_DAYS = 1     -- enables Time Travel for 1 day
    COMMENT = 'ShopFlow Analytics Platform — Olist Brazilian E-Commerce dataset';

-- Set this database as the default for this session
USE DATABASE SHOPFLOW_DB;

-- Confirm database was created and Time Travel is enabled
SHOW DATABASES LIKE 'SHOPFLOW%';

In [ ]:
-- Verify full session context — warehouse + database should both show
SELECT
    CURRENT_WAREHOUSE() AS active_warehouse,
    CURRENT_DATABASE()  AS active_database,
    CURRENT_ROLE()      AS active_role;

---
## Cell 4 — Create a role and grant privileges

### What is a role in Snowflake?

A **role** is a collection of privileges. Instead of granting privileges directly to users, you grant them to roles, then assign roles to users. This is the same concept as IAM roles in AWS — you attach policies to roles, then attach roles to users or services.

```text
AWS IAM:                        Snowflake RBAC:
Policy → IAM Role → User        Privileges → Role → User
```

### Snowflake's built-in role hierarchy

Snowflake ships with five system roles arranged in a hierarchy — higher roles inherit all privileges of lower ones:

```text
ACCOUNTADMIN          ← most powerful — full account control
    └── SYSADMIN      ← creates and manages all objects (warehouses, databases, tables)
    └── SECURITYADMIN ← manages users, roles, and grants
            └── USERADMIN  ← creates users and roles only
                    └── PUBLIC  ← default role every user has — minimal privileges
```

### Why create a custom role?

Best practice is to **never do day-to-day work as ACCOUNTADMIN**. It is like never logging into AWS as root. Instead:
- Use `SYSADMIN` for creating objects (what we are doing in this setup notebook)
- Use a custom role like `SHOPFLOW_ROLE` for running queries and loading data

This way, if something goes wrong in a load script, it cannot accidentally modify account-level settings.

> **🛡️ Security Note: RBAC Rules**
> 
> Role-Based Access Control (RBAC) is central to Snowflake security. Key things to keep in mind:
> - `GRANT PRIVILEGE ON OBJECT TO ROLE` — gives a role permission on a specific object
> - `GRANT ROLE TO USER` — assigns a role to a user
> - `WITH GRANT OPTION` — allows the recipient to further grant that privilege to others
> - **Object ownership:** whoever creates an object **owns** it by default. The owner role has full control.

In [ ]:
-- Switch to SECURITYADMIN for role and grant management
USE ROLE SECURITYADMIN;

-- Create a custom role for the ShopFlow project
CREATE ROLE IF NOT EXISTS SHOPFLOW_ROLE
    COMMENT = 'Role for ShopFlow project — data loading and querying';

-- Grant the new role to your user so you can USE ROLE SHOPFLOW_ROLE
-- Replace YOUR_USERNAME with your actual Snowflake username
GRANT ROLE SHOPFLOW_ROLE TO USER YOUR_USERNAME;

-- Give SHOPFLOW_ROLE the ability to use the warehouse
GRANT USAGE ON WAREHOUSE SHOPFLOW_WH TO ROLE SHOPFLOW_ROLE;

-- Give SHOPFLOW_ROLE full access to the database
GRANT OWNERSHIP ON DATABASE SHOPFLOW_DB TO ROLE SHOPFLOW_ROLE COPY CURRENT GRANTS;

-- Switch to SYSADMIN to verify everything looks correct
USE ROLE SYSADMIN;

-- Confirm the role exists
SHOW ROLES LIKE 'SHOPFLOW%';

In [ ]:
-- Confirm grants on the warehouse
SHOW GRANTS ON WAREHOUSE SHOPFLOW_WH;

---
## Cell 5 — Create three schemas

### What is a schema?

A schema is a **namespace** — a logical container inside a database that groups related objects together. Every table, stage, file format, and view lives inside exactly one schema.

The full path to any object in Snowflake is always:
```text
database_name . schema_name . object_name
e.g.  SHOPFLOW_DB . SHOPFLOW_RAW . RAW_ORDERS
```

This three-part name is called a **fully qualified name**. When you have set a default database and schema with `USE DATABASE` and `USE SCHEMA`, you can refer to objects by just their name. But in scripts that run across multiple schemas, always use the fully qualified name to avoid ambiguity.

### Why three schemas — one per layer?

We create one schema for each layer of our warehouse architecture:

| Schema | Layer | What lives here | Rule |
|--------|-------|-----------------|------|
| `SHOPFLOW_RAW` | Raw | Tables with source data exactly as received | Never modify or delete rows |
| `SHOPFLOW_STAGED` | Staged | Type-cast, deduplicated, validated tables | Source of truth for transforms |
| `SHOPFLOW_ANALYTICS` | Analytics | Star schema — fact + dimension tables | What analysts and dashboards query |

Three schemas instead of one gives you:
- **Access control** — grant analysts `SELECT` on ANALYTICS only, without exposing raw data
- **Clarity** — anyone opening Snowsight immediately knows which layer they are in
- **Blast radius control** — you can drop and recreate STAGED without touching RAW or ANALYTICS

> **💡 Did You Know?**
> 
> When you create a database, Snowflake automatically creates two schemas inside it: `PUBLIC` (default schema for objects with no explicit schema) and `INFORMATION_SCHEMA` (read-only system views about your objects — like `INFORMATION_SCHEMA.TABLES`, `INFORMATION_SCHEMA.COLUMNS`). You will see these in your `SHOW SCHEMAS` output.

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE SHOPFLOW_DB;

-- Layer 1: RAW schema — the landing zone
-- Data arrives here exactly as it came from the source — no transformations
-- Golden rule: rows in RAW are never updated or deleted
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_RAW
    DATA_RETENTION_TIME_IN_DAYS = 1
    COMMENT = 'Landing zone — raw source data, never modified. All columns VARCHAR.';

-- Layer 2: STAGED schema — the cleaning room
-- RAW data is cast to proper types, deduplicated, and NULLs are handled here
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_STAGED
    DATA_RETENTION_TIME_IN_DAYS = 1
    COMMENT = 'Cleaned and typed data — ready for transformation into analytics layer.';

-- Layer 3: ANALYTICS schema — the business layer
-- Star schema lives here — what analysts and dashboards actually query
CREATE SCHEMA IF NOT EXISTS SHOPFLOW_ANALYTICS
    DATA_RETENTION_TIME_IN_DAYS = 1
    COMMENT = 'Star schema — fact and dimension tables for business analysis.';

-- Confirm all schemas created
-- You will also see PUBLIC and INFORMATION_SCHEMA — these are auto-created by Snowflake
SHOW SCHEMAS IN DATABASE SHOPFLOW_DB;

---
## Cell 6 — Create a file format

### What is a file format?

A **file format** is a named, reusable object that tells Snowflake exactly how to parse incoming files. When Snowflake reads a CSV file during `COPY INTO`, it needs to know:
- What character separates fields? (comma, tab, pipe?)
- Are fields enclosed in quotes? Which quote character?
- Should it skip the header row?
- What strings should be treated as NULL?
- What text encoding is used?

You could specify all this inline on every `COPY INTO` statement. But we have 9 files to load. Instead, define it once as a named object and reference it by name — cleaner, safer, easier to change.

### Every option explained

| Option | Value | Why |
|--------|-------|-----|
| `TYPE = CSV` | CSV | Tells Snowflake which parser to use |
| `FIELD_OPTIONALLY_ENCLOSED_BY = '"'` | Double quote | A field like `"São Paulo, SP"` contains a comma — wrapping it in quotes tells Snowflake this comma is part of the value, not a separator |
| `SKIP_HEADER = 1` | 1 row | The first row of every CSV is the column header — skip it, do not load it as data |
| `NULL_IF = ('NULL','null','','NA','N/A')` | List of strings | Any of these string values will be stored as SQL `NULL` instead of the literal text |
| `EMPTY_FIELD_AS_NULL = TRUE` | True | Two consecutive commas `,,` means an empty field — store as `NULL`, not as an empty string |
| `ENCODING = UTF8` | UTF-8 | The Olist dataset has Portuguese characters: ã, ç, é, õ. UTF-8 handles them correctly — ASCII would corrupt them |

> **📐 Design Tip: Object Scopes**
> 
> File formats can be created at three scopes: **account level**, **database level**, or **schema level**. We create ours inside `SHOPFLOW_RAW` (schema level) because it is specific to our raw data ingestion pipeline. A format at account level would be shared across all databases in the account.

In [ ]:
-- All ingestion objects (file format, stage, raw tables) live in SHOPFLOW_RAW
USE SCHEMA SHOPFLOW_RAW;

-- Create the CSV file format for all 9 Olist CSV files
CREATE OR REPLACE FILE FORMAT OLIST_CSV_FORMAT
    TYPE                         = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'              -- handles commas inside quoted fields
    SKIP_HEADER                  = 1                -- skip the column header row
    NULL_IF                      = ('NULL', 'null', '', 'NA', 'N/A')
    EMPTY_FIELD_AS_NULL          = TRUE             -- empty field becomes NULL not ''
    ENCODING                     = UTF8             -- Portuguese characters need UTF-8
    COMMENT                      = 'CSV parser for Olist Brazilian E-Commerce dataset';

-- Verify the file format and inspect all its properties
SHOW FILE FORMATS IN SCHEMA SHOPFLOW_RAW;

In [ ]:
-- You can also describe it for a cleaner view
DESCRIBE FILE FORMAT OLIST_CSV_FORMAT;

---
## Cell 7 — Create an internal stage

### What is a stage?

A **stage** is a named location where files are held temporarily before being loaded into Snowflake tables. It is the loading dock of your data warehouse.

The two-step loading process works like this:

```text
Step 1 — PUT:         local file  →  stage  (upload)
Step 2 — COPY INTO:   stage       →  table  (load)
```

You cannot load directly from a local file into a table in one step. The stage is always the intermediate stop.

### Internal vs external stage

There are two types of stages:

| | Internal stage | External stage |
|--|----------------|----------------|
| Storage managed by | Snowflake | You (S3 / GCS / Azure Blob) |
| Credentials needed | None | Yes — storage integration or IAM |
| Setup complexity | None — just CREATE STAGE | Needs cloud bucket + permissions |
| Good for | Learning, simple loads | Production pipelines, Snowpipe |
| Cost | Included in Snowflake storage billing | Your cloud storage costs |

We use an **internal stage** for this project — no external cloud setup required.

### How to reference a stage in SQL

Stages are always referenced with the `@` prefix:
```sql
@OLIST_STAGE                     -- shorthand (when schema is already set)
@SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_STAGE  -- fully qualified name
```

Useful stage commands:
```sql
LIST @OLIST_STAGE;               -- show all files currently in the stage
REMOVE @OLIST_STAGE/myfile.csv;  -- delete a specific file from the stage
```

> **📦 Concept Note: Types of Internal Stages**
> 
> There are actually three types of internal stages: **named stages** (what we are creating here — a reusable, named object), **user stages** (one per user, referenced as `@~`), and **table stages** (one per table, referenced as `@%table_name`). Named stages are the most flexible and the most commonly used in production.

In [ ]:
-- Create the internal stage inside SHOPFLOW_RAW schema
-- This is where our Python ingestion code will PUT the 9 Olist CSV files
-- FILE_FORMAT attached here means COPY INTO inherits it automatically
CREATE OR REPLACE STAGE OLIST_STAGE
    FILE_FORMAT = OLIST_CSV_FORMAT
    COMMENT     = 'Internal stage for Olist CSV files — intermediate stop between Kaggle and RAW tables';

-- Confirm the stage was created
SHOW STAGES IN SCHEMA SHOPFLOW_RAW;

In [ ]:
-- List files in the stage — should be empty right now
LIST @OLIST_STAGE;

In [ ]:
-- Check what we have built so far in SHOPFLOW_RAW
SHOW OBJECTS IN SCHEMA SHOPFLOW_RAW;

---
## Cells 8 – 16 — Create nine RAW tables

### Read this before running any table cell

**Why are all columns VARCHAR?**

Every column in every RAW table is `VARCHAR`. This is intentional and is a real-world best practice used by data teams at large companies.

The RAW layer has one job: **receive data without losing any of it**. If a date column in the source file occasionally contains `'N/A'` or a malformed value like `'2023-99-99'`, a `TIMESTAMP` or `DATE` column would reject that row. Your `COPY INTO` would either fail or skip the row. With `VARCHAR`, every row loads — bad values and all.

Type casting, validation, and cleaning all happen in `SHOPFLOW_STAGED`. That is the right place to handle data quality — where you have full control and can log and handle exceptions explicitly.

**Why `_loaded_at` and `_source_file` on every table?**

These two metadata columns are added to every RAW table:

| Column | Type | Value | Purpose |
|--------|------|-------|--------|
| `_loaded_at` | `TIMESTAMP_NTZ` | Auto-populated by `DEFAULT CURRENT_TIMESTAMP()` | Tells you exactly when each row was loaded — essential for debugging and incremental load logic |
| `_source_file` | `VARCHAR` | Populated manually in `COPY INTO` | Tells you which CSV file a row came from — data lineage tracing |

The `_` prefix is a naming convention that signals to every engineer: *this is system metadata, not business data from the source*.

**`TIMESTAMP_NTZ` — what does NTZ mean?**

Snowflake has three timestamp types:
- `TIMESTAMP_NTZ` — No Time Zone. Stores the value as-is, no timezone conversion. Best for DWH tables — store everything in UTC, convert at reporting layer.
- `TIMESTAMP_LTZ` — Local Time Zone. Converts to and from the session timezone.
- `TIMESTAMP_TZ` — Time Zone stored with the value.

Use `TIMESTAMP_NTZ` for all warehouse tables. It is the industry-standard recommended type for data warehousing.

**`CREATE OR REPLACE` vs `CREATE IF NOT EXISTS`**

We use `CREATE OR REPLACE` during setup — it drops and recreates the table if it already exists. This is safe right now because tables are empty. Once you have loaded data, **do not re-run these cells** — `CREATE OR REPLACE` will delete your data. At that point use `CREATE TABLE IF NOT EXISTS` instead.

---
## Cell 8 — RAW_ORDERS

The central table of the entire dataset. Every other table connects back to orders through `order_id`. One row = one order placed by a customer.

**Key business concept — order lifecycle:**
```text
created → approved → processing → shipped → delivered → cancelled / unavailable
```
The four timestamp columns (`purchase`, `approved`, `carrier`, `customer`) track this journey. The gap between `order_estimated_delivery_date` and `order_delivered_customer_date` measures delivery performance — a key metric for the analytics layer.

**Important Note:** `customer_id` here is NOT a unique person identifier. One real customer who places three orders will have three different `customer_id` values. The truly unique identifier for a person is `customer_unique_id` in the customers table. This is a critical distinction to note when analyzing this dataset.

In [ ]:
USE SCHEMA SHOPFLOW_RAW;

CREATE OR REPLACE TABLE RAW_ORDERS (
    order_id                          VARCHAR,  -- unique order identifier — primary join key
    customer_id                       VARCHAR,  -- links to RAW_CUSTOMERS — NOT unique per person
    order_status                      VARCHAR,  -- created/approved/processing/shipped/delivered/cancelled
    order_purchase_timestamp          VARCHAR,  -- when customer placed the order
    order_approved_at                 VARCHAR,  -- when payment was approved by the platform
    order_delivered_carrier_date      VARCHAR,  -- when seller handed the package to the carrier
    order_delivered_customer_date     VARCHAR,  -- when customer actually received the package
    order_estimated_delivery_date     VARCHAR,  -- delivery date promised to the customer at purchase
    -- metadata: auto-populated at load time, not from the source CSV
    _loaded_at                        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file                      VARCHAR
);

SELECT 'RAW_ORDERS created — 8 business columns + 2 metadata columns' AS status;

---
## Cell 9 — RAW_ORDER_ITEMS

The line-item table — one row per item within an order. Since a customer can order multiple products in one order, this table can have more rows than `RAW_ORDERS`.

The unique key for a row here is the **combination** of `order_id + order_item_id`, not either field alone.

An important real-world detail about the Olist marketplace: different items in the same order can come from different sellers. Customer orders one item from a seller in São Paulo and another from a seller in Rio — both items are in the same order, but `seller_id` is different for each.

In [ ]:
CREATE OR REPLACE TABLE RAW_ORDER_ITEMS (
    order_id             VARCHAR,  -- FK to RAW_ORDERS
    order_item_id        VARCHAR,  -- item sequence within the order: 1, 2, 3...
    product_id           VARCHAR,  -- FK to RAW_PRODUCTS
    seller_id            VARCHAR,  -- FK to RAW_SELLERS — different per item within same order
    shipping_limit_date  VARCHAR,  -- deadline for seller to hand package to carrier
    price                VARCHAR,  -- item price in Brazilian Reais (BRL)
    freight_value        VARCHAR,  -- shipping cost for this item in BRL
    _loaded_at           TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file         VARCHAR
);

SELECT 'RAW_ORDER_ITEMS created — 7 business columns + 2 metadata columns' AS status;

---
## Cell 10 — RAW_CUSTOMERS

Customer location data. The `customer_id` vs `customer_unique_id` distinction is critical — it trips up many people working with this dataset for the first time.

- **`customer_id`** — generated fresh for every order. One person who places 5 orders will appear in this table 5 times with 5 different `customer_id` values. This is how Olist anonymised the data.
- **`customer_unique_id`** — the real unique person. Use this when you want to count how many distinct customers the business has.

The `customer_zip_code_prefix` (first 5 digits of the zip) links to `RAW_GEOLOCATION` for latitude and longitude — useful for geographic delivery analysis.

In [ ]:
CREATE OR REPLACE TABLE RAW_CUSTOMERS (
    customer_id              VARCHAR,  -- per-order ID — NOT unique per person
    customer_unique_id       VARCHAR,  -- true unique person identifier
    customer_zip_code_prefix VARCHAR,  -- first 5 digits of zip — links to RAW_GEOLOCATION
    customer_city            VARCHAR,
    customer_state           VARCHAR,  -- 2-letter Brazilian state code: SP, RJ, MG, RS...
    _loaded_at               TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file             VARCHAR
);

SELECT 'RAW_CUSTOMERS created — 5 business columns + 2 metadata columns' AS status;

---
## Cell 11 — RAW_SELLERS

One row per seller registered on the Olist platform. Sellers are the small businesses that list their products for sale.

Only ~3,000 rows — a small dimension table. In the analytics star schema, this becomes `DIM_SELLERS`. Seller location (`seller_state`) is useful for analysing whether proximity between seller and customer affects delivery time and review scores.

In [ ]:
CREATE OR REPLACE TABLE RAW_SELLERS (
    seller_id              VARCHAR,  -- unique seller identifier
    seller_zip_code_prefix VARCHAR,  -- links to RAW_GEOLOCATION for seller coordinates
    seller_city            VARCHAR,
    seller_state           VARCHAR,  -- 2-letter Brazilian state code
    _loaded_at             TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file           VARCHAR
);

SELECT 'RAW_SELLERS created — 4 business columns + 2 metadata columns' AS status;

---
## Cell 12 — RAW_PRODUCTS

One row per product. Contains the category (in Portuguese) and physical dimensions.

Two things to notice:
1. `product_category_name` is in Portuguese (e.g. `cama_mesa_banho`). The `RAW_CATEGORY_TRANSLATION` table maps these to English. We join the two in STAGED.
2. The source CSV has a typo: `product_name_lenght` and `product_description_lenght` (should be `length`). We **preserve the typo in RAW** — RAW is an exact copy of the source. Fixing column names is a transformation that happens in STAGED.

In [ ]:
CREATE OR REPLACE TABLE RAW_PRODUCTS (
    product_id                  VARCHAR,  -- unique product identifier
    product_category_name       VARCHAR,  -- category in Portuguese — join to RAW_CATEGORY_TRANSLATION
    product_name_lenght         VARCHAR,  -- typo preserved from source — fixed in STAGED layer
    product_description_lenght  VARCHAR,  -- typo preserved from source — fixed in STAGED layer
    product_photos_qty          VARCHAR,  -- number of product listing photos
    product_weight_g            VARCHAR,  -- weight in grams
    product_length_cm           VARCHAR,
    product_height_cm           VARCHAR,
    product_width_cm            VARCHAR,
    _loaded_at                  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file                VARCHAR
);

SELECT 'RAW_PRODUCTS created — 9 business columns + 2 metadata columns' AS status;

---
## Cell 13 — RAW_PAYMENTS

One row per payment event per order. An order can have multiple payment rows — for example a customer might pay part with a voucher and the rest by credit card. Sum `payment_value` grouped by `order_id` to get the total order value.

**Payment types in Brazilian e-commerce:**
- `credit_card` — most common. `payment_installments` tells you how many months they split it across.
- `boleto` — a Brazilian bank payment slip. Pay at any bank, lottery house, or ATM. Very common in Brazil.
- `voucher` — store credit or gift voucher.
- `debit_card` — direct debit.

In [ ]:
CREATE OR REPLACE TABLE RAW_PAYMENTS (
    order_id              VARCHAR,  -- FK to RAW_ORDERS
    payment_sequential    VARCHAR,  -- payment number for this order: 1, 2, 3...
    payment_type          VARCHAR,  -- credit_card / boleto / voucher / debit_card
    payment_installments  VARCHAR,  -- number of monthly instalments (1 = paid in full)
    payment_value         VARCHAR,  -- amount for this payment row in BRL
    _loaded_at            TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file          VARCHAR
);

SELECT 'RAW_PAYMENTS created — 5 business columns + 2 metadata columns' AS status;

---
## Cell 14 — RAW_REVIEWS

Customer satisfaction data. One row per review. Many customers only leave a star score without writing a comment — so `review_comment_title` and `review_comment_message` will be NULL for a large portion of rows.

`review_score` (1–5 stars) is one of the most analytically valuable columns in the dataset. In the analytics layer we will calculate average review score by seller, by product category, by delivery delay, and by state.

In [ ]:
CREATE OR REPLACE TABLE RAW_REVIEWS (
    review_id                VARCHAR,  -- unique review identifier
    order_id                 VARCHAR,  -- FK to RAW_ORDERS
    review_score             VARCHAR,  -- star rating: 1 to 5
    review_comment_title     VARCHAR,  -- optional short title in Portuguese (often NULL)
    review_comment_message   VARCHAR,  -- optional free-text in Portuguese (often NULL)
    review_creation_date     VARCHAR,  -- when Olist sent the review request to customer
    review_answer_timestamp  VARCHAR,  -- when the customer actually submitted their review
    _loaded_at               TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file             VARCHAR
);

SELECT 'RAW_REVIEWS created — 7 business columns + 2 metadata columns' AS status;

---
## Cell 15 — RAW_GEOLOCATION

Maps Brazilian zip code prefixes to latitude, longitude, city, and state. The largest table — over 1 million rows — because many zip codes have multiple coordinate entries.

This table enriches both customers and sellers with geographic coordinates. The join key is `geolocation_zip_code_prefix` → `customer_zip_code_prefix` (in customers) or `seller_zip_code_prefix` (in sellers).

**Important data quality issue to be aware of:** there are duplicate rows for the same zip prefix with slightly different lat/lng values. This is a known quirk of this dataset. We will deduplicate in STAGED using a window function to keep one coordinate per zip prefix.

In [ ]:
CREATE OR REPLACE TABLE RAW_GEOLOCATION (
    geolocation_zip_code_prefix  VARCHAR,  -- join key to customer and seller zip codes
    geolocation_lat              VARCHAR,  -- latitude coordinate
    geolocation_lng              VARCHAR,  -- longitude coordinate
    geolocation_city             VARCHAR,
    geolocation_state            VARCHAR,  -- 2-letter Brazilian state code
    _loaded_at                   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file                 VARCHAR
);

SELECT 'RAW_GEOLOCATION created — 5 business columns + 2 metadata columns' AS status;

---
## Cell 16 — RAW_CATEGORY_TRANSLATION

A tiny lookup table — only 71 rows. Maps Portuguese product category names to English.

The Olist dataset is Brazilian, so all product categories in `RAW_PRODUCTS` are in Portuguese. This table provides translations. In STAGED we join the two tables to add the English name to every product.

Examples of what you will find here:
- `cama_mesa_banho` → `bed_bath_table`
- `esporte_lazer` → `sports_leisure`
- `informatica_acessorios` → `computers_accessories`
- `moveis_decoracao` → `furniture_decor`

In [ ]:
CREATE OR REPLACE TABLE RAW_CATEGORY_TRANSLATION (
    product_category_name          VARCHAR,  -- Portuguese name — joins to RAW_PRODUCTS
    product_category_name_english  VARCHAR,  -- English translation
    _loaded_at                     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _source_file                   VARCHAR
);

SELECT 'RAW_CATEGORY_TRANSLATION created — 2 business columns + 2 metadata columns' AS status;

---
## Cell 17 — Final verification

Run these final cells. They query `INFORMATION_SCHEMA` — Snowflake's built-in catalogue of all your objects — to confirm everything was created correctly.

*(We've split these into individual cells so you can see the result grid for every single query!)*

### What is INFORMATION_SCHEMA?

Every Snowflake database automatically contains a special read-only schema called `INFORMATION_SCHEMA`. It holds system views that describe everything in your database — tables, columns, stages, file formats, load history, and more. You query it with regular SQL.

> **🔍 Pro Tip: INFORMATION_SCHEMA vs ACCOUNT_USAGE**
> 
> `INFORMATION_SCHEMA` is real-time — queries return the current state instantly. Contrast this with `SNOWFLAKE.ACCOUNT_USAGE` views which have a latency of up to 45 minutes but retain history for up to a year and cover the entire account (not just one database). It's important to know when to use each one based on your reporting needs.

In [ ]:
USE DATABASE SHOPFLOW_DB;

-- 1. Confirm warehouse exists and is running
SHOW WAREHOUSES LIKE 'SHOPFLOW%';

In [ ]:
-- 2. Confirm all 5 schemas exist (RAW, STAGED, ANALYTICS + PUBLIC + INFORMATION_SCHEMA)
SHOW SCHEMAS IN DATABASE SHOPFLOW_DB;

In [ ]:
-- 3. Confirm file format exists in SHOPFLOW_RAW
SHOW FILE FORMATS IN SCHEMA SHOPFLOW_RAW;

In [ ]:
-- 4. Confirm stage exists in SHOPFLOW_RAW
SHOW STAGES IN SCHEMA SHOPFLOW_RAW;

In [ ]:
-- 5. Confirm all 9 RAW tables exist
-- row_count = 0 for all is correct — data has not been loaded yet
SELECT
    table_name                                      AS table_name,
    table_type                                      AS table_type,
    row_count                                       AS rows_loaded,
    bytes                                           AS bytes_stored,
    TO_CHAR(CONVERT_TIMEZONE('UTC', created),
            'YYYY-MM-DD HH24:MI')                  AS created_at,
    table_owner                                     AS owned_by
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_RAW'
  AND table_name   LIKE 'RAW_%'
ORDER BY table_name;

In [ ]:
-- 6. Check the column count per RAW table
SELECT
    table_name,
    COUNT(*) AS column_count
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'SHOPFLOW_RAW'
  AND table_name   LIKE 'RAW_%'
GROUP BY table_name
ORDER BY table_name;

In [ ]:
-- 7. Summary count — should show exactly 9 tables
SELECT COUNT(*) AS total_raw_tables
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_RAW'
  AND table_name LIKE 'RAW_%';

---
## 🛠 Self-Guided Exploration:

Before moving on to the data loading phase, prove that your environment is ready and test your understanding of the concepts covered.

Try to solve these 4 challenges by writing the SQL commands from scratch in the empty cells below. Do not just scroll up and copy!

### Challenge 1: Verify Your Context
**Objective:** Ensure you understand how to check your active session parameters.

**Task:** Write a single `SELECT` statement that returns your current active role, warehouse, database, and schema.

In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2: Inspect the Internal Stage
**Objective:** Practice interacting with stage objects before loading data.

**Task:** Write the Snowflake command to list all files currently inside the `OLIST_STAGE` stage.

> 💡 **Hint:** The `LIST` command takes a stage reference prefixed with `@`. You can use just `@OLIST_STAGE` if the active schema is already set, or the fully-qualified path `@SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_STAGE` which works regardless of active schema — good habit for production scripts.

> ⏳ The stage should be empty right now — that is the correct and expected result at this point.

In [ ]:
-- Write your SQL for Challenge 2 here


### Challenge 3: Query the Information Schema
**Objective:** Master the metadata dictionary.

**Task:** Write a query against `SHOPFLOW_DB.INFORMATION_SCHEMA.COLUMNS` to find the `data_type` of the `_loaded_at` column specifically for the `RAW_ORDERS` table.

> 💡 **Hint:** `INFORMATION_SCHEMA.COLUMNS` holds one row per column across *every* table in the database. If you filter only by `column_name = '_loaded_at'` you will get 9 rows back — one for every RAW table, since all 9 share that metadata column. You need **two** WHERE conditions to get a single clean result: one to pin the table, one to pin the column.

> 🎯 **SnowPro tip:** The three columns to know in `INFORMATION_SCHEMA.COLUMNS` are `table_schema`, `table_name`, and `column_name` — they form the composite key that uniquely identifies any column in your database.

In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4: Security and Roles Validation
**Objective:** Verify that the custom role you created exists in the account.

**Task:** Write the `SHOW` command that lists all roles in your account, and optionally filter it using `LIKE` to only show roles starting with 'SHOPFLOW'.

In [ ]:
-- Write your SQL for Challenge 4 here


---
## Setup complete

If Cell 17 shows 1 warehouse, 5 schemas, 1 file format, 1 stage, and 9 tables with 0 rows — your Snowflake environment is fully ready for data loading.

---

### Full object tree — what you just built

```text
your Snowflake account
├── SHOPFLOW_WH                         (virtual warehouse — X-Small, auto-suspend 60s)
├── SHOPFLOW_ROLE                       (custom role — for project work)
└── SHOPFLOW_DB                         (database)
    ├── SHOPFLOW_RAW                    (schema — landing zone)
    │   ├── OLIST_CSV_FORMAT            (file format — how to parse CSVs)
    │   ├── OLIST_STAGE                 (internal stage — loading dock)
    │   ├── RAW_ORDERS                  (~99k rows after load)
    │   ├── RAW_ORDER_ITEMS             (~112k rows after load)
    │   ├── RAW_CUSTOMERS               (~99k rows after load)
    │   ├── RAW_SELLERS                 (~3k rows after load)
    │   ├── RAW_PRODUCTS                (~33k rows after load)
    │   ├── RAW_PAYMENTS                (~104k rows after load)
    │   ├── RAW_REVIEWS                 (~99k rows after load)
    │   ├── RAW_GEOLOCATION             (~1M rows after load)
    │   └── RAW_CATEGORY_TRANSLATION    (~71 rows after load)
    ├── SHOPFLOW_STAGED                 (empty — built in Phase 2)
    └── SHOPFLOW_ANALYTICS              (empty — built in Phase 3)
```

---

### What to do next — Data Loading

The next part of this project uses **Python cells (Snowpark)** inside a Snowflake Notebook. It will:

1. Download the 9 Olist CSV files from Kaggle using the Kaggle API
2. `PUT` each file into `@SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_STAGE`
3. `COPY INTO` each RAW table from the stage
4. Verify row counts match expected values
5. Run data quality checks to understand what you loaded

---

### Key concepts you learned in this notebook

| Concept | What it means |
|---------|---------------|
| Compute ≠ storage | Warehouse = CPU/memory. Database = data. Completely separate. |
| Object hierarchy | account → warehouse/database → schema → table/stage/format |
| RBAC | Privileges → Role → User. Never work as ACCOUNTADMIN day-to-day. |
| Three-layer architecture | RAW (exact copy) → STAGED (clean + typed) → ANALYTICS (star schema) |
| All-VARCHAR RAW | Never reject data at ingestion. Type cast in STAGED. |
| Internal stage | Snowflake-managed loading dock. PUT uploads here. COPY INTO reads from here. |
| INFORMATION_SCHEMA | Real-time catalogue of all objects in your database. |
| TIMESTAMP_NTZ | No timezone. Standard choice for data warehouse tables. |